# Synthetic Oversampling Quality Evaluation for WSN Intrusion Dataset

This notebook evaluates how well different oversampling methods (SMOTE variants and ADASYN, plus SMOTE-ENN and SMOTE-Tomek) generate synthetic samples for minority classes.

We compute state-of-the-art and practical metrics to assess similarity and support coverage of synthetic data relative to the real data:

- Distributional similarity
  - Maximum Mean Discrepancy (MMD, RBF kernels)
  - Energy Distance (multivariate)
  - Sliced Wasserstein Distance (multi-projection)
  - Fréchet Distance (Gaussian approximation, FID-style)
- Support/coverage metrics (PRDC): Precision, Recall, Density, Coverage
- Two-sample classifier test (AUC): how well a classifier can distinguish real vs synthetic

Design notes:
- Metrics are computed per class that was oversampled and summarized overall.
- Distance-based metrics operate on a stratified subsample for tractability on large data.
- Features are standardized using statistics from real samples only.
- ID-like columns are excluded from metrics (e.g., id, who CH) to avoid artifacts.

Outputs:
- A results DataFrame by method and class
- Optional JSON export to `models/wsn_intrusion_detection/artifacts/`

In [1]:
# Setup: imports, versions, and lightweight dependency checks
import sys, json, math, os, random, warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# Optional third-party packages used for metrics
# We'll attempt imports and provide fallbacks if unavailable.

try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.neighbors import NearestNeighbors
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score
    from sklearn.ensemble import RandomForestClassifier
except Exception as e:
    raise RuntimeError("scikit-learn is required. Please install scikit-learn.")

# SciPy distances
try:
    from scipy.spatial.distance import cdist, pdist, squareform
except Exception:
    raise RuntimeError("SciPy is required for distance metrics. Please install scipy.")

# tqdm (optional)
try:
    from tqdm import tqdm as _tqdm
    tqdm = _tqdm
except Exception:
    def tqdm(x):
        return x

# PRDC metrics helper (precision, recall, density, coverage)
# Adapted from common implementations using k-NN radii.

def compute_knn_radii(X: np.ndarray, k: int = 5) -> np.ndarray:
    nn = NearestNeighbors(n_neighbors=min(k+1, len(X))).fit(X)
    dists, _ = nn.kneighbors(X)
    radii = dists[:, -1]  # distance to k-th neighbor
    return radii


def prdc_metrics(real_features: np.ndarray, fake_features: np.ndarray, k: int = 5) -> Dict[str, float]:
    # Based on https://arxiv.org/abs/1904.06991
    if len(real_features) < 2 or len(fake_features) < 2:
        return {m: np.nan for m in ["precision", "recall", "density", "coverage"]}

    k = max(1, min(k, len(real_features)-1, len(fake_features)-1))

    real_radii = compute_knn_radii(real_features, k=k)
    fake_radii = compute_knn_radii(fake_features, k=k)

    real_nn = NearestNeighbors(n_neighbors=1).fit(real_features)
    fake_nn = NearestNeighbors(n_neighbors=1).fit(fake_features)

    # Precision: fraction of fake within real neighborhood
    d_rf, _ = real_nn.kneighbors(fake_features)
    precision = float(np.mean((d_rf[:, 0] <= real_radii).astype(float)))

    # Recall: fraction of real within fake neighborhood
    d_fr, _ = fake_nn.kneighbors(real_features)
    recall = float(np.mean((d_fr[:, 0] <= fake_radii).astype(float)))

    # Density: average number of real neighbors within real k-radius around fake, normalized by k
    Rk = np.expand_dims(real_radii, 0)
    nnR = NearestNeighbors(n_neighbors=min(k, len(real_features))).fit(real_features)
    d_fk, _ = nnR.kneighbors(fake_features)
    density = float(np.mean((d_fk <= Rk[:, :d_fk.shape[1]]).sum(axis=1) / k))

    # Coverage: fraction of real points that have at least one fake within their real k-radius
    nnF = NearestNeighbors(n_neighbors=1).fit(fake_features)
    d_rf2, _ = nnF.kneighbors(real_features)
    coverage = float(np.mean((d_rf2[:, 0] <= real_radii).astype(float)))

    return {"precision": precision, "recall": recall, "density": density, "coverage": coverage}


# Energy distance (multivariate) using sample estimator
# E^2 = 2E||X-Y|| - E||X-X'|| - E||Y-Y'||

def energy_distance(X: np.ndarray, Y: np.ndarray, metric: str = "euclidean") -> float:
    if len(X) == 0 or len(Y) == 0:
        return np.nan
    XY = cdist(X, Y, metric=metric)
    XX = pdist(X, metric=metric)
    YY = pdist(Y, metric=metric)
    e = 2 * XY.mean() - XX.mean() - YY.mean()
    return float(max(e, 0.0))


# RBF MMD with median heuristic for bandwidth(s)

def pairwise_sq_dists(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    return ((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=2)


def mmd_rbf(X: np.ndarray, Y: np.ndarray, gammas: List[float] = None) -> float:
    if len(X) == 0 or len(Y) == 0:
        return np.nan
    Z = np.vstack([X, Y])
    if Z.shape[0] > 2000:  # subsample for bandwidth
        idx = np.random.choice(Z.shape[0], size=2000, replace=False)
        Zb = Z[idx]
    else:
        Zb = Z
    pdZ = pdist(Zb, metric="euclidean")
    med = np.median(pdZ)
    if med <= 0:
        med = 1.0
    if gammas is None:
        # Use multiple bandwidths for stability
        sigmas = np.array([med/2, med, 2*med])
        gammas = 1.0 / (2.0 * (sigmas ** 2))

    XX = pairwise_sq_dists(X, X)
    YY = pairwise_sq_dists(Y, Y)
    XY = pairwise_sq_dists(X, Y)

    def ksum(D, gamma):
        K = np.exp(-gamma * D)
        return (K.sum() - np.trace(K)) / (len(D) * (len(D) - 1) + 1e-12)

    mmd2 = 0.0
    for g in gammas:
        kxx = ksum(XX, g)
        kyy = ksum(YY, g)
        kxy = np.exp(-g * XY).mean()
        mmd2 += kxx + kyy - 2 * kxy
    mmd2 /= len(gammas)
    return float(max(mmd2, 0.0))


# Sliced Wasserstein distance (SWD) via random projections

def sliced_wasserstein(X: np.ndarray, Y: np.ndarray, n_projections: int = 64, seed: int = 42) -> float:
    if len(X) == 0 or len(Y) == 0:
        return np.nan
    rng = np.random.default_rng(seed)
    d = X.shape[1]
    projs = rng.normal(size=(n_projections, d))
    projs /= np.linalg.norm(projs, axis=1, keepdims=True) + 1e-12
    sw = []
    for w in projs:
        xw = X @ w
        yw = Y @ w
        xw = np.sort(xw)
        yw = np.sort(yw)
        n = min(len(xw), len(yw))
        sw.append(np.mean(np.abs(xw[:n] - yw[:n])))
    return float(np.mean(sw))


# Fréchet distance between Gaussians (FID-style)
from numpy.linalg import norm


def frechet_distance_gaussian(X: np.ndarray, Y: np.ndarray, eps: float = 1e-6) -> float:
    if len(X) < 2 or len(Y) < 2:
        return np.nan
    mx, my = X.mean(axis=0), Y.mean(axis=0)
    Cx = np.cov(X, rowvar=False)
    Cy = np.cov(Y, rowvar=False)
    Cx += eps * np.eye(Cx.shape[0])
    Cy += eps * np.eye(Cy.shape[0])
    evals, evecs = np.linalg.eigh(Cx @ Cy)
    evals = np.clip(evals, 0, None)
    sqrt_prod = evecs @ np.diag(np.sqrt(evals)) @ evecs.T
    diff = mx - my
    fid = diff @ diff + np.trace(Cx + Cy - 2 * sqrt_prod)
    return float(max(fid, 0.0))


# Two-sample classifier test (AUC)

def two_sample_auc(real: np.ndarray, fake: np.ndarray, seed: int = 42) -> float:
    if len(real) == 0 or len(fake) == 0:
        return np.nan
    X = np.vstack([real, fake])
    y = np.array([0] * len(real) + [1] * len(fake))
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    clf = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)
    clf.fit(Xtr, ytr)
    prob = clf.predict_proba(Xte)[:, 1]
    return float(roc_auc_score(yte, prob))


# Global config
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

DATA_PATH = "data/WSN-DS.csv"
ARTIFACT_DIR = "models/wsn_intrusion_detection/artifacts"
RESULTS_JSON = os.path.join(ARTIFACT_DIR, "synthetic_quality_metrics.json")

print("Environment ready. Versions:")
print({
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
})

Environment ready. Versions:
{'python': '3.9.6', 'numpy': '2.0.2', 'pandas': '2.3.0+4.g1dfc98e16a'}


In [2]:
# Load dataset and prepare features/labels
raw = pd.read_csv(DATA_PATH)
print("Raw shape:", raw.shape)
print("Columns:", list(raw.columns))

# Identify label column
label_col = None
candidates = ["Attack type", "attack_type", "label", "target", "y"]
for c in candidates:
    if c in raw.columns:
        label_col = c
        break
if label_col is None:
    raise ValueError("Could not find label column. Expected one of: " + ", ".join(candidates))

# Remove likely identifier columns (string or index-like):
exclude_cols = set([label_col])
for col in raw.columns:
    lc = col.strip().lower()
    if lc in {"id", "who ch", "who_ch", "time", "send_code"}:
        exclude_cols.add(col)

# Keep numeric features only (for distances) and drop excluded columns
num_cols = [c for c in raw.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(raw[c])]
X_all = raw[num_cols].copy()
y_all = raw[label_col].astype(str).copy()

print(f"Selected {len(num_cols)} numeric feature columns for metrics.")

# Train/test split to avoid data leakage when testing two-sample classifier.
# We use only the training portion of real data to fit scalers and generate synthetics.
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=RANDOM_SEED, stratify=y_all)
print("Train shape:", X_train.shape)

# Standardize using real train statistics
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train.values)
X_test_s = scaler.transform(X_test.values)

# Create per-class index mapping
classes = sorted(y_train.unique())
class_indices_train = {c: np.where(y_train.values == c)[0] for c in classes}
class_counts = {c: int((y_train == c).sum()) for c in classes}
print("Class counts (train):", class_counts)

# Determine minority class(es)
min_count = min(class_counts.values())
maj_count = max(class_counts.values())
minority_classes = [c for c, n in class_counts.items() if n == min_count or n < 0.2 * maj_count]
print("Minority classes for oversampling:", minority_classes if minority_classes else classes)

Raw shape: (374661, 19)
Columns: [' id', ' Time', ' Is_CH', ' who CH', ' Dist_To_CH', ' ADV_S', ' ADV_R', ' JOIN_S', ' JOIN_R', ' SCH_S', ' SCH_R', 'Rank', ' DATA_S', ' DATA_R', ' Data_Sent_To_BS', ' dist_CH_To_BS', ' send_code ', 'Expaned Energy', 'Attack type']
Selected 14 numeric feature columns for metrics.
Train shape: (299728, 14)
Class counts (train): {'Blackhole': 8039, 'Flooding': 2650, 'Grayhole': 11677, 'Normal': 272052, 'TDMA': 5310}
Minority classes for oversampling: ['Blackhole', 'Flooding', 'Grayhole', 'TDMA']
Class counts (train): {'Blackhole': 8039, 'Flooding': 2650, 'Grayhole': 11677, 'Normal': 272052, 'TDMA': 5310}
Minority classes for oversampling: ['Blackhole', 'Flooding', 'Grayhole', 'TDMA']


In [3]:
# Oversampling configurations and generation helpers
from collections import defaultdict

try:
    from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
    from imblearn.combine import SMOTEENN, SMOTETomek
except Exception:
    raise RuntimeError("imbalanced-learn is required. Please install imbalanced-learn.")


def make_strategy(balanced_to: int, y: pd.Series, only_classes: List[str]) -> Dict[str, int]:
    strat = {}
    for c in only_classes:
        n = int((y == c).sum())
        if n < balanced_to:
            strat[c] = balanced_to
    return strat


# Define the oversamplers we want to evaluate
oversamplers = {
    "SMOTE": lambda strat: SMOTE(random_state=RANDOM_SEED, sampling_strategy=strat, k_neighbors=min(5, max(1, min([sum(y_train==c) for c in strat.keys()]) - 1)) if strat else 3),
    "BorderlineSMOTE": lambda strat: BorderlineSMOTE(random_state=RANDOM_SEED, sampling_strategy=strat, kind="borderline-1"),
    "ADASYN": lambda strat: ADASYN(random_state=RANDOM_SEED, sampling_strategy=strat, n_neighbors=5),
}

# Combine methods operate on full dataset, not per-class custom strat easily
combine_methods = {
    "SMOTEENN": SMOTEENN(random_state=RANDOM_SEED),
    "SMOTETomek": SMOTETomek(random_state=RANDOM_SEED),
}

# Target balance level (e.g., to 0.8 of majority count to avoid excessive synthesis)
TARGET_BALANCE = max(int(0.8 * maj_count), min_count + 1)
print("Target per-minority count:", TARGET_BALANCE)

# Generate synthetic samples for each oversampler (class-wise view)
synthetics = defaultdict(dict)  # method -> class -> np.ndarray (standardized)

# Fit on training data only
for name, ctor in oversamplers.items():
    strat = make_strategy(TARGET_BALANCE, y_train, minority_classes or classes)
    if not strat:
        continue
    sampler = ctor(strat)
    X_res, y_res = sampler.fit_resample(X_train, y_train)
    # Find newly added samples for each class
    # Approach: count per class before/after, then identify indices of samples labeled class exceeding original count
    X_res = pd.DataFrame(X_res, columns=X_train.columns)
    y_res = pd.Series(y_res)
    for cls in strat.keys():
        n_before = int((y_train == cls).sum())
        idx_cls = np.where(y_res.values == cls)[0]
        # Take the last (len(idx_cls) - n_before) as synthetic approximation
        n_synth = max(0, len(idx_cls) - n_before)
        if n_synth > 0:
            synth_idx = idx_cls[-n_synth:]
            X_synth = X_res.iloc[synth_idx].values
            X_synth_s = scaler.transform(X_synth)
            synthetics[name][cls] = X_synth_s
        else:
            synthetics[name][cls] = np.empty((0, X_train.shape[1]))
    print(f"Generated synthetics for {name}")

# For combine methods, collect all synthetic minority samples by diffing size
for name, sampler in combine_methods.items():
    X_res, y_res = sampler.fit_resample(X_train, y_train)
    X_res = pd.DataFrame(X_res, columns=X_train.columns)
    y_res = pd.Series(y_res)
    # Estimate which samples are synthetic: For each minority class, find extra beyond original count and treat tail as synthetic
    for cls in (minority_classes or classes):
        n_before = int((y_train == cls).sum())
        idx_cls = np.where(y_res.values == cls)[0]
        n_synth = max(0, len(idx_cls) - n_before)
        if n_synth > 0:
            synth_idx = idx_cls[-n_synth:]
            X_synth = X_res.iloc[synth_idx].values
            X_synth_s = scaler.transform(X_synth)
            synthetics[name][cls] = X_synth_s
        else:
            synthetics[name][cls] = np.empty((0, X_train.shape[1]))
    print(f"Generated synthetics for {name}")

# Materialize real per-class standardized samples for metric comparisons
real_std_by_class = {c: X_train_s[class_indices_train[c]] for c in classes}

print("Oversampling completed.")

Target per-minority count: 217641
Generated synthetics for SMOTE
Generated synthetics for SMOTE
Generated synthetics for BorderlineSMOTE
Generated synthetics for BorderlineSMOTE
Generated synthetics for ADASYN
Generated synthetics for ADASYN
Generated synthetics for SMOTEENN
Generated synthetics for SMOTEENN
Generated synthetics for SMOTETomek
Oversampling completed.
Generated synthetics for SMOTETomek
Oversampling completed.


In [4]:
# Compute metrics per method and per class

MAX_PER_CLASS = 2000  # cap for distance metrics
N_PROJECTIONS = 64
K_PRDC = 5

rows = []

for method, cls_map in tqdm(synthetics.items()):
    for cls, X_fake_s in cls_map.items():
        X_real_s = real_std_by_class.get(cls, np.empty((0, X_train_s.shape[1])))
        if len(X_real_s) == 0 or len(X_fake_s) == 0:
            rows.append({
                "method": method,
                "class": cls,
                "n_real": int(len(X_real_s)),
                "n_fake": int(len(X_fake_s)),
                "mmd_rbf": np.nan,
                "energy": np.nan,
                "swd": np.nan,
                "fid": np.nan,
                "prdc_precision": np.nan,
                "prdc_recall": np.nan,
                "prdc_density": np.nan,
                "prdc_coverage": np.nan,
                "two_sample_auc": np.nan,
            })
            continue
        # Subsample for metrics
        n = min(MAX_PER_CLASS, len(X_real_s), len(X_fake_s))
        idx_r = np.random.choice(len(X_real_s), size=n, replace=False)
        idx_f = np.random.choice(len(X_fake_s), size=n, replace=False)
        R = X_real_s[idx_r]
        F = X_fake_s[idx_f]

        # Metrics
        mmd = mmd_rbf(R, F)
        eng = energy_distance(R, F)
        swd = sliced_wasserstein(R, F, n_projections=N_PROJECTIONS, seed=RANDOM_SEED)
        fid = frechet_distance_gaussian(R, F)
        prdc = prdc_metrics(R, F, k=K_PRDC)
        auc = two_sample_auc(R, F)

        rows.append({
            "method": method,
            "class": cls,
            "n_real": int(len(X_real_s)),
            "n_fake": int(len(X_fake_s)),
            "mmd_rbf": mmd,
            "energy": eng,
            "swd": swd,
            "fid": fid,
            "prdc_precision": prdc["precision"],
            "prdc_recall": prdc["recall"],
            "prdc_density": prdc["density"],
            "prdc_coverage": prdc["coverage"],
            "two_sample_auc": auc,
        })

results_df = pd.DataFrame(rows)

# Aggregate summary across classes per method
agg = results_df.groupby("method").agg({
    "mmd_rbf": "mean",
    "energy": "mean",
    "swd": "mean",
    "fid": "mean",
    "prdc_precision": "mean",
    "prdc_recall": "mean",
    "prdc_density": "mean",
    "prdc_coverage": "mean",
    "two_sample_auc": "mean",
}).reset_index().sort_values(["two_sample_auc", "mmd_rbf"], ascending=[True, True])

print("Per-class results (head):")
display(results_df.head())
print("\nAggregated summary (lower is better for distances and AUC; higher is better for PRDC):")
display(agg)

# Save artifacts
os.makedirs(ARTIFACT_DIR, exist_ok=True)
results_payload = {
    "metadata": {
        "dataset": os.path.basename(DATA_PATH),
        "features": num_cols,
        "label": label_col,
        "timestamp": pd.Timestamp.utcnow().isoformat(),
        "max_per_class": MAX_PER_CLASS,
        "n_projections": N_PROJECTIONS,
        "k_prdc": K_PRDC,
    },
    "per_class": results_df.to_dict(orient="records"),
    "summary": agg.to_dict(orient="records"),
}
with open(RESULTS_JSON, "w") as f:
    json.dump(results_payload, f, indent=2)
print(f"Saved metrics to {RESULTS_JSON}")

  0%|          | 0/5 [00:00<?, ?it/s]/var/folders/rd/_1y3pdfn5t59bv5mghnzrx5r0000gn/T/ipykernel_11774/3618610859.py:148: RuntimeWarning: divide by zero encountered in matmul
  xw = X @ w
/var/folders/rd/_1y3pdfn5t59bv5mghnzrx5r0000gn/T/ipykernel_11774/3618610859.py:148: RuntimeWarning: overflow encountered in matmul
  xw = X @ w
/var/folders/rd/_1y3pdfn5t59bv5mghnzrx5r0000gn/T/ipykernel_11774/3618610859.py:148: RuntimeWarning: invalid value encountered in matmul
  xw = X @ w
/var/folders/rd/_1y3pdfn5t59bv5mghnzrx5r0000gn/T/ipykernel_11774/3618610859.py:149: RuntimeWarning: divide by zero encountered in matmul
  yw = Y @ w
/var/folders/rd/_1y3pdfn5t59bv5mghnzrx5r0000gn/T/ipykernel_11774/3618610859.py:149: RuntimeWarning: overflow encountered in matmul
  yw = Y @ w
/var/folders/rd/_1y3pdfn5t59bv5mghnzrx5r0000gn/T/ipykernel_11774/3618610859.py:149: RuntimeWarning: invalid value encountered in matmul
  yw = Y @ w
/var/folders/rd/_1y3pdfn5t59bv5mghnzrx5r0000gn/T/ipykernel_11774/3618610859.p

Per-class results (head):


,method,class,n_real,n_fake,mmd_rbf,energy,swd,fid,prdc_precision,prdc_recall,prdc_density,prdc_coverage,two_sample_auc
0,SMOTE,Blackhole,8039,209602,0.000117,0.000666,0.029301,0.771767,0.7145,0.7075,0.5745,0.9870,0.519599
1,SMOTE,Flooding,2650,214991,0.000729,0.005817,0.073802,0.768973,0.8115,0.7200,0.7197,0.9750,0.507121
2,SMOTE,Grayhole,11677,205964,0.000000,0.000000,0.032803,0.000000,0.6440,0.6260,0.7199,0.9805,0.500031
3,SMOTE,TDMA,5310,212331,0.000000,0.000000,0.050541,0.000000,0.7380,0.7060,0.4544,0.9875,0.505869
4,BorderlineSMOTE,Blackhole,8039,209602,0.166273,1.046246,0.439357,10.135956,0.4250,0.2065,0.4925,0.0925,0.989783



Aggregated summary (lower is better for distances and AUC; higher is better for PRDC):


,method,mmd_rbf,energy,swd,fid,prdc_precision,prdc_recall,prdc_density,prdc_coverage,two_sample_auc
3,SMOTEENN,0.000238,0.003336,0.067279,0.108330,0.731875,0.686000,0.569175,0.977750,0.496132
4,SMOTETomek,0.000227,0.003995,0.062580,0.417128,0.740250,0.696625,0.621700,0.983875,0.505427
2,SMOTE,0.000211,0.001621,0.046612,0.385185,0.727000,0.689875,0.617125,0.982500,0.508155
0,ADASYN,0.127677,1.839843,0.790363,26.347553,0.482875,0.433375,0.365300,0.300750,0.924019
1,BorderlineSMOTE,0.162889,2.492642,0.959218,30.366615,0.483750,0.286500,0.351675,0.207625,0.960326


Saved metrics to models/wsn_intrusion_detection/artifacts/synthetic_quality_metrics.json


## Interpreting the metrics

- MMD/Energy/SWD/FID: lower is better (synthetic closer to real). Values are not on the same scale; compare methods relative to each other.
- PRDC (Precision/Recall/Density/Coverage): higher is better. Precision reflects fidelity; Recall/Coverage reflect support coverage; Density balances mode-collapse vs over-dispersion.
- Two-sample AUC: lower is better (0.5 is ideal, meaning classifier cannot tell synthetic from real). Much higher than 0.7 suggests detectable artifacts.

Notes:
- All metrics computed on standardized numeric features and per-class subsets.
- Results are averaged across classes; inspect per-class rows to spot specific weaknesses.

Next steps:
- Tune k for SMOTE variants per minority size.
- Try BorderlineSMOTE-2, KMeansSMOTE, or SVMSMOTE if available.
- Evaluate impact on downstream model performance (already in your modeling notebook) and combine with these quality metrics.